# AI-Driven Stock Market Prediction and Strategic Portfolio Management
### Coupled Modular Deep Learning (CMLF) + Temporal-Dynamic Reliability Fusion (TDRF) + Adaptive Fuzzy Min–Max Decision Network (AFMM-D)

Full implementation of the 10-stage methodology:

| Stage | Content | Notebook section |
|---|---|---|
| 1 | Dataset collection and temporal preparation (Yahoo Finance, FRED, NSE) | §1 |
| 2 | Preprocessing and feature construction | §2 |
| 3 | **CMLF** – Coupled Modular Learning Framework | §3 |
| 4 | **TDRF** – Temporal-Dynamic Reliability Fusion | §4 |
| 5 | **AFMM-D** – Adaptive Fuzzy Min–Max Market Decision Network | §5 |
| 6 | Multi-target prediction (direction, return, volatility, confidence) | §6 |
| 7 | Risk-aware BUY / HOLD / SELL signals and position sizing | §7 |
| 8 | Strategic portfolio management | §8 |
| 9 | Robustness and ablation | §9 |
| 10 | Quantitative and financial evaluation | §10 |

**Notes**
* Set `MARKET = 'US' | 'NSE' | 'BOTH'` in the config cell. `FAST_MODE=True` gives a quick smoke-test run.
* If data download fails (no internet), the notebook falls back to a *synthetic regime-switching market* purely so the pipeline can be executed end to end. **Never report synthetic results.**
* The four existing methods (BGSTO-DC, MOPO, AI-MDP, MBi-LSTM) are implemented as **architecture-family proxies** trained with the identical multi-target protocol and the identical signal/portfolio engine. Replace `BaselineNet` bodies with the exact architectures from refs [16]–[19] for the final paper.
* Research code only, not investment advice.

In [ ]:
# ============================ 0. SETUP ============================
!pip install -q yfinance scikit-learn matplotlib tqdm
import os, time, math, random, warnings, copy, json
from types import SimpleNamespace as NS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)

FAST_MODE = False                 # True -> fewer tickers / epochs (smoke test)
MARKET    = 'US'                  # 'US' | 'NSE' | 'BOTH'
USE_SYNTHETIC_FALLBACK = True     # only used if downloads fail
SEED = 42

TICKERS = {'US':  ['AAPL','MSFT','JPM','XOM','JNJ','AMZN','PG','NVDA'],
           'NSE': ['RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','ICICIBANK.NS','ITC.NS','LT.NS','SBIN.NS']}
CTX = {'US':  dict(idx=['^GSPC','^IXIC','^DJI'],       vix='^VIX',      fred=['SP500','NASDAQCOM','DJIA'], fred_vix='VIXCLS'),
       'NSE': dict(idx=['^NSEI','^NSEBANK','^BSESN'],  vix='^INDIAVIX', fred=None, fred_vix=None)}

cfg = NS(
    start='2015-01-01', end='2025-12-31',     # FRED SP500/NASDAQ/DJIA/VIX only cover ~10 years
    window=30, horizon=5, split=(0.70, 0.15, 0.15), dir_thr_mult=0.30,
    # model
    d=32, e=16, drop=0.1, batch=256, epochs=40, patience=8, lr=1e-3, wd=1e-4, lam_ret=0.5, lam_vol=0.5,
    # AFMM-D
    afmm_theta_grid=(0.45, 0.65, 0.85), afmm_epochs=3, afmm_gamma=6.0, afmm_tau=0.7, beta_mix=0.7,
    # signals / portfolio
    conf_q=0.40, ra_min=0.0, target_vol=1.0, base_size=0.30, vol_hi_q=0.80,
    max_w=0.35, max_gross=1.0, rebalance_every=5, adj_band=0.05,
    dd_trigger=0.10, pvol_limit=0.30, preserve_scale=0.5, cost_bps=10,
    noise_levels=(0.0, 0.05, 0.10, 0.20, 0.40), cost_grid=(0, 5, 10, 25, 50),
)
if FAST_MODE:
    TICKERS = {k: v[:4] for k, v in TICKERS.items()}
    cfg.epochs, cfg.patience, cfg.afmm_theta_grid, cfg.afmm_epochs = 8, 4, (0.65,), 2

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

---
## §1  Stage 1 — Stock Market Dataset Collection and Temporal Preparation
Yahoo Finance (OHLCV + Adj Close) → FRED / index market indicators (S&P 500, NASDAQ, DJIA, VIX) → NSE equities & indices (via Yahoo `.NS` tickers; swap in `nsepy`/NSE bhavcopy if preferred) → chronological alignment → return sequences → chronological split → market-context integration.

In [ ]:
# ===================== 1. DATA COLLECTION =====================
def yf_download(sym):
    import yfinance as yf
    df = yf.download(sym, start=cfg.start, end=cfg.end, auto_adjust=False, progress=False)
    if df is None or len(df) == 0: raise ValueError('empty download: ' + sym)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df

def fred_series(sid):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
    s = pd.read_csv(url, na_values='.', parse_dates=[0], index_col=0).iloc[:, 0].astype(float)
    return s[(s.index >= cfg.start) & (s.index <= cfg.end)]

def load_context(market):
    # market indicators: FRED first (US), Yahoo index tickers as fallback / NSE source
    c = CTX[market]; cols = {}
    for k, ysym in enumerate(c['idx']):
        s = None
        if c['fred']:
            try: s = fred_series(c['fred'][k])
            except Exception: s = None
        if s is None or s.dropna().shape[0] < 500:
            s = yf_download(ysym)['Close']
        cols[f'ctx{k+1}'] = s
    v = None
    if c['fred_vix']:
        try: v = fred_series(c['fred_vix'])
        except Exception: v = None
    if v is None or v.dropna().shape[0] < 500:
        v = yf_download(c['vix'])['Close']
    cols['vix'] = v
    return pd.concat(cols, axis=1).sort_index()

def synthetic_market(n_stocks, seed=0):
    # regime-switching synthetic market (calm / turbulent) with weak momentum -> pipeline smoke test ONLY
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(cfg.start, cfg.end); n = len(dates)
    reg = np.zeros(n, int)
    for t in range(1, n):
        reg[t] = reg[t-1] if rng.random() > (0.02 if reg[t-1] == 0 else 0.06) else 1 - reg[t-1]
    sig = np.where(reg == 0, 0.007, 0.020); mu = np.where(reg == 0, 0.0004, -0.0006)
    m = mu + sig * rng.standard_normal(n)
    ctx = pd.DataFrame(index=dates)
    for k, b in enumerate([1.0, 1.2, 0.9]):
        r = b * m + 0.002 * rng.standard_normal(n)
        ctx[f'ctx{k+1}'] = 3000 * np.exp(np.cumsum(r))
    ctx['vix'] = pd.Series(sig * np.sqrt(252) * 100, index=dates).ewm(span=5).mean() + rng.standard_normal(n)
    stocks = {}
    for i in range(n_stocks):
        beta = rng.uniform(0.7, 1.4); r = np.zeros(n)
        idio = rng.uniform(0.008, 0.016) * (1 + 0.5 * reg)
        for t in range(1, n):
            r[t] = beta * m[t] + idio[t] * rng.standard_normal() + 0.06 * r[t-1] - 0.02 * np.sign(r[t-2] if t > 1 else 0) * abs(r[t-1])
        c = 100 * np.exp(np.cumsum(r)); o = np.r_[c[0], c[:-1]] * np.exp(0.002 * rng.standard_normal(n))
        h = np.maximum(o, c) * np.exp(np.abs(0.004 * rng.standard_normal(n)) * (1 + reg))
        l = np.minimum(o, c) * np.exp(-np.abs(0.004 * rng.standard_normal(n)) * (1 + reg))
        v = 1e6 * np.exp(0.3 * rng.standard_normal(n) + 0.5 * reg + 0.3 * np.abs(r) / 0.01)
        stocks[f'SYN{i}'] = pd.DataFrame(dict(Open=o, High=h, Low=l, Close=c, **{'Adj Close': c}, Volume=v), index=dates)
    return stocks, ctx

def collect_raw():
    markets = ['US', 'NSE'] if MARKET == 'BOTH' else [MARKET]
    raw, used_synth = {}, False
    try:
        for mk in markets:
            ctx = load_context(mk)
            for t in TICKERS[mk]:
                df = yf_download(t)[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
                raw[t] = df.join(ctx, how='left')           # Chronological Data Alignment (trading dates of the stock)
    except Exception as ex:
        if not USE_SYNTHETIC_FALLBACK: raise
        print('!! DOWNLOAD FAILED (%s) -> using SYNTHETIC data. Do NOT report these results.' % str(ex)[:80])
        stocks, ctx = synthetic_market(len(sum([TICKERS[m] for m in markets], [])), SEED)
        raw = {k: v.join(ctx, how='left') for k, v in stocks.items()}; used_synth = True
    out = {}
    for t, df in raw.items():
        df = df.sort_index()
        df[['ctx1', 'ctx2', 'ctx3', 'vix']] = df[['ctx1', 'ctx2', 'ctx3', 'vix']].ffill(limit=3)
        df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Adj Close', 'ctx1', 'ctx2', 'ctx3', 'vix'])
        df = df[df['Volume'] >= 0]
        # Return Sequence Construction
        df['ret'] = df['Adj Close'].pct_change()
        df['logret'] = np.log(df['Adj Close']).diff()
        out[t] = df
    return out, used_synth

RAW, USED_SYNTH = collect_raw()
TICK = list(RAW.keys())
print('Tickers:', TICK, '| synthetic =', USED_SYNTH)
print({t: (d.index[0].date(), d.index[-1].date(), len(d)) for t, d in RAW.items()})
RAW[TICK[0]].tail(3)

---
## §2  Stage 2 — Financial Data Preprocessing and Feature Construction
Missing-value treatment (forward-fill, chronology preserved) → outlier treatment (**winsorisation with train-period percentiles**, genuine extremes are kept, only clipped at the 0.5/99.5 % level) → technical indicators (RSI, MACD, SMA, EMA, Bollinger, ATR) → volatility features → train-fitted normalisation → sliding windows → temporal alignment of the three feature groups (**price / technical / volatility**).

Targets (all known only after the window end): `y_ret` = next-day log-return (%), `y_vol` = realised volatility of the following `horizon` days (%), `y_dir` ∈ {0 down, 1 stable, 2 up} using a dead-zone `±0.3·σ_train`.
The chronological split is applied to *dates* (same cut-offs for every stock), and the last `horizon` samples of train/val are purged so no target window crosses a split boundary.

In [ ]:
# ===================== 2. PREPROCESSING =====================
def rsi(c, n=14):
    d = c.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    ru = up.ewm(alpha=1/n, adjust=False, min_periods=n).mean(); rd = dn.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + ru / (rd + 1e-12))
def ema(x, n): return x.ewm(span=n, adjust=False, min_periods=n).mean()
def atr(h, l, c, n=14):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/n, adjust=False, min_periods=n).mean()

PRICE_F = ['lr', 'gap', 'oc', 'hl', 'dlogv', 'm1', 'm2', 'm3']
TECH_F  = ['rsi', 'macd', 'macd_sig', 'macd_hist', 'sma10', 'sma30', 'ema12', 'ema26', 'bb_pb', 'bb_w', 'atr']
VOL_F   = ['rv5', 'rv10', 'rv20', 'rng10', 'vrat', 'vstd10', 'vix_lvl', 'vix_chg', 'm1_rv10']

def build_features(df):
    f = df['Adj Close'] / df['Close']                        # split/dividend adjustment factor
    o, h, l, c = df['Open'] * f, df['High'] * f, df['Low'] * f, df['Adj Close']
    lr = df['logret']; lv = np.log1p(df['Volume'])
    m = {k: np.log(df[k]).diff() for k in ['ctx1', 'ctx2', 'ctx3']}
    P = pd.DataFrame({'lr': lr, 'gap': np.log(o / c.shift()), 'oc': np.log(c / o), 'hl': np.log(h / l),
                      'dlogv': lv.diff(), 'm1': m['ctx1'], 'm2': m['ctx2'], 'm3': m['ctx3']})
    macd = ema(c, 12) - ema(c, 26); sig = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    sma20, sd20 = c.rolling(20).mean(), c.rolling(20).std()
    T = pd.DataFrame({'rsi': rsi(c) / 100, 'macd': macd / c, 'macd_sig': sig / c, 'macd_hist': (macd - sig) / c,
                      'sma10': c.rolling(10).mean() / c - 1, 'sma30': c.rolling(30).mean() / c - 1,
                      'ema12': ema(c, 12) / c - 1, 'ema26': ema(c, 26) / c - 1,
                      'bb_pb': (c - (sma20 - 2 * sd20)) / (4 * sd20 + 1e-12), 'bb_w': 4 * sd20 / (sma20 + 1e-12),
                      'atr': atr(h, l, c) / c})
    V = pd.DataFrame({'rv5': lr.rolling(5).std(), 'rv10': lr.rolling(10).std(), 'rv20': lr.rolling(20).std(),
                      'rng10': np.log(h / l).rolling(10).mean(), 'vrat': lv - np.log1p(df['Volume'].rolling(20).mean()),
                      'vstd10': lv.rolling(10).std(), 'vix_lvl': np.log(df['vix'] + 1e-6), 'vix_chg': np.log(df['vix'] + 1e-6).diff(),
                      'm1_rv10': m['ctx1'].rolling(10).std()})
    y_ret = lr.shift(-1) * 100
    y_vol = lr.rolling(cfg.horizon).std().shift(-cfg.horizon) * 100
    feats = pd.concat([P, T, V], axis=1).replace([np.inf, -np.inf], np.nan)
    feats = feats.ffill(limit=5)                                              # Missing-Value Treatment
    return feats, y_ret, y_vol

FEATS = {}
for t in TICK:
    F_, yr, yv = build_features(RAW[t])
    ok = F_.notna().all(axis=1)
    FEATS[t] = (F_[ok], yr[ok], yv[ok], RAW[t]['vix'][ok])

all_dates = np.array(sorted(set().union(*[set(v[0].index) for v in FEATS.values()])))
n_d = len(all_dates)
TR_END = all_dates[int(cfg.split[0] * n_d) - 1]
VA_END = all_dates[int((cfg.split[0] + cfg.split[1]) * n_d) - 1]
print('Train: <= %s | Val: <= %s | Test: after' % (str(TR_END)[:10], str(VA_END)[:10]))

def make_windows(arr, T):
    w = np.lib.stride_tricks.sliding_window_view(arr, T, axis=0)           # (N-T+1, F, T)
    return np.ascontiguousarray(w.transpose(0, 2, 1)).astype(np.float32)

def build_split_data():
    T, H = cfg.window, cfg.horizon
    parts = {'train': [], 'val': [], 'test': []}; info = {}
    for tid, t in enumerate(TICK):
        F_, yr, yv, vix = FEATS[t]
        dates = F_.index.values; L = len(F_)
        itr = np.searchsorted(dates, TR_END, side='right') - 1          # last row index in train
        iva = np.searchsorted(dates, VA_END, side='right') - 1
        trm = np.arange(L) <= itr
        # Outlier treatment: winsorise at train-period 0.5/99.5 percentiles (extremes kept, influence reduced)
        lo, hi = F_[trm].quantile(0.005), F_[trm].quantile(0.995)
        X = F_.clip(lo, hi, axis=1)
        mu, sd = X[trm].mean(), X[trm].std() + 1e-8                     # Price & Feature Normalization (train only)
        X = ((X - mu) / sd).clip(-8, 8)
        Wp = make_windows(X[PRICE_F].values, T); Wt = make_windows(X[TECH_F].values, T); Wv = make_windows(X[VOL_F].values, T)
        idx = np.arange(T - 1, L)                                       # last row of every window
        yr_, yv_ = yr.values[idx], yv.values[idx]
        valid = np.isfinite(yr_) & np.isfinite(yv_)
        thr = cfg.dir_thr_mult * np.nanstd(yr.values[:itr + 1]); info[t] = thr
        ydir = np.where(yr_ < -thr, 0, np.where(yr_ > thr, 2, 1)).astype(np.int64)
        seg = np.where(idx <= itr - H, 'train', np.where((idx > itr) & (idx <= iva - H), 'val', np.where(idx > iva, 'test', 'purge')))
        for s in parts:
            mk = valid & (seg == s)
            parts[s].append(dict(Xp=Wp[mk], Xt=Wt[mk], Xv=Wv[mk], y_dir=ydir[mk], y_ret=yr_[mk].astype(np.float32),
                                 y_vol=yv_[mk].astype(np.float32), tid=np.full(mk.sum(), tid), date=dates[idx][mk],
                                 vix_raw=vix.values[idx][mk].astype(np.float32)))
    data = {}
    for s, lst in parts.items():
        data[s] = {k: np.concatenate([p[k] for p in lst]) for k in lst[0]}
    return data, info

DATA, DIR_THR = build_split_data()
for s in DATA: print(s, {k: v.shape for k, v in DATA[s].items() if k in ('Xp', 'Xt', 'Xv')}, 'N=', len(DATA[s]['y_dir']))
print('Class balance (train) [down, stable, up]:', np.bincount(DATA['train']['y_dir'], minlength=3) / len(DATA['train']['y_dir']))
DIMS = (len(PRICE_F), len(TECH_F), len(VOL_F))
CLASS_W = torch.tensor(len(DATA['train']['y_dir']) / (3 * np.bincount(DATA['train']['y_dir'], minlength=3)), dtype=torch.float32)
VOL_HI = float(np.quantile(DATA['train']['y_vol'], cfg.vol_hi_q))          # high-volatility threshold (train only)
REG_Q = (float(np.quantile(DATA['train']['vix_raw'], 0.5)), float(np.quantile(DATA['train']['vix_raw'], 0.8)))

---
## §3  Stage 3 — CMLF: Coupled Modular Learning Framework
Three specialised encoders (price dynamics = multi-scale Conv1d + GRU, technical indicators = nonlinear MLP + GRU, volatility context = GRU) → **cross-module coupling** (each module cross-attends to the other two, gated residual) → **coupled feature interaction** (pairwise multiplicative interactions) → **adaptive feature fusion** (softmax gate).
Output: per-step module representations `G (B,T,3,d)` and the unified coupled representation `U (B,T,d)`.

In [ ]:
# ===================== 3. CMLF =====================
class ModuleEncoder(nn.Module):
    def __init__(self, in_dim, d, kind, drop=0.1):
        super().__init__()
        self.kind = kind
        if kind == 'price':                                   # Price Dynamics Module (short + medium range patterns)
            self.c3 = nn.Conv1d(in_dim, d // 2, 3, padding=1)
            self.c5 = nn.Conv1d(in_dim, d // 2, 5, padding=2)
        elif kind == 'tech':                                  # Technical Indicator Module (nonlinear indicator relations)
            self.inp = nn.Sequential(nn.Linear(in_dim, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
        else:                                                 # Volatility Context Module
            self.inp = nn.Linear(in_dim, d)
        self.gru = nn.GRU(d, d, batch_first=True)             # Temporal Feature Extraction
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(drop)
    def forward(self, x):
        if self.kind == 'price':
            z = x.transpose(1, 2); z = F.gelu(torch.cat([self.c3(z), self.c5(z)], 1).transpose(1, 2))
        else:
            z = self.inp(x)
        h, _ = self.gru(z)
        return self.norm(self.drop(h) + z)                    # Independent Feature Encoding

class CrossModuleCoupling(nn.Module):
    def __init__(self, d, heads=4, drop=0.1):
        super().__init__()
        self.attn = nn.ModuleList([nn.MultiheadAttention(d, heads, batch_first=True, dropout=drop) for _ in range(3)])
        self.gate = nn.ModuleList([nn.Linear(2 * d, d) for _ in range(3)])
        self.norm = nn.ModuleList([nn.LayerNorm(d) for _ in range(3)])
    def forward(self, H):
        out = []
        for i in range(3):
            others = torch.cat([H[j] for j in range(3) if j != i], 1)       # complementary information from the other modules
            c, _ = self.attn[i](H[i], others, others)
            g = torch.sigmoid(self.gate[i](torch.cat([H[i], c], -1)))
            out.append(self.norm[i](H[i] + g * c))
        return out

class CoupledInteraction(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.pairs = [(0, 1), (0, 2), (1, 2)]                 # price x technical, price x volatility, technical x volatility
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU()) for _ in self.pairs])
        self.norm = nn.ModuleList([nn.LayerNorm(d) for _ in range(3)])
    def forward(self, H):
        inter = [torch.zeros_like(H[0]) for _ in range(3)]
        for p, (i, j) in enumerate(self.pairs):
            z = self.proj[p](H[i] * H[j]); inter[i] = inter[i] + z; inter[j] = inter[j] + z
        return [self.norm[i](H[i] + inter[i] / 2) for i in range(3)]

class CMLF(nn.Module):
    def __init__(self, dims, d, coupling=True, drop=0.1):
        super().__init__()
        self.coupling = coupling
        self.enc = nn.ModuleList([ModuleEncoder(dims[0], d, 'price', drop), ModuleEncoder(dims[1], d, 'tech', drop), ModuleEncoder(dims[2], d, 'vol', drop)])
        if coupling: self.cross = CrossModuleCoupling(d, drop=drop); self.inter = CoupledInteraction(d)
        self.fuse = nn.Linear(d, 1)                           # Adaptive Feature Fusion
    def forward(self, xp, xt, xv):
        H = [self.enc[0](xp), self.enc[1](xt), self.enc[2](xv)]
        if self.coupling: H = self.inter(self.cross(H))       # Cross-Module Coupling + Coupled Feature Interaction
        G = torch.stack(H, 2)                                 # (B,T,3,d)
        a = torch.softmax(self.fuse(G).squeeze(-1), -1)
        U = (a.unsqueeze(-1) * G).sum(2)                      # (B,T,d)
        return G, U

---
## §4  Stage 4 — TDRF: Temporal-Dynamic Reliability Fusion
Temporal context encoder (GRU) → market-state characterisation (soft assignment to `K` learnable state prototypes) → per-group **reliability estimation** from (group features, context, state) → **temporal reliability updating** (learnable exponential smoothing of reliability logits over the window) → dynamic weights → cross-feature self-attention → compact **market embedding** (bounded to [0,1] so it can be consumed by the fuzzy hyperboxes).
`fusion='concat'` and `fusion='fixed'` are the ablation replacements used in §9.

In [ ]:
# ===================== 4. TDRF =====================
class TDRF(nn.Module):
    def __init__(self, d, e, K=4, mode='dynamic', drop=0.1):
        super().__init__()
        self.mode = mode; self.d = d
        self.ctx = nn.GRU(d, d, batch_first=True)                              # Temporal Context Encoding
        self.protos = nn.Parameter(torch.randn(K, d) * 0.1)                    # Market-State Characterization
        self.rel = nn.Sequential(nn.Linear(3 * d, d), nn.GELU(), nn.Linear(d, 1))   # Feature Reliability Estimation
        self.alpha = nn.Parameter(torch.zeros(3))                              # Temporal Reliability Updating (smoothing rate)
        self.fixed = nn.Parameter(torch.zeros(3))                              # fixed-weight ablation
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True, dropout=drop)    # Cross-Feature Interaction
        self.norm = nn.LayerNorm(d)
        in_dim = 5 * d if mode == 'dynamic' else 3 * d
        self.out = nn.Sequential(nn.Linear(in_dim, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, e))   # Unified Feature Fusion
    def forward(self, G, U):
        B, T, _, d = G.shape
        if self.mode == 'dynamic':
            c, _ = self.ctx(U)                                                   # (B,T,d)
            s = torch.softmax(c @ self.protos.t() / math.sqrt(d), -1)            # (B,T,K) state assignment
            m = s @ self.protos                                                  # (B,T,d) state embedding
            z = self.rel(torch.cat([G, c.unsqueeze(2).expand(-1, -1, 3, -1), m.unsqueeze(2).expand(-1, -1, 3, -1)], -1)).squeeze(-1)  # (B,T,3)
            a = torch.sigmoid(self.alpha); zs = [z[:, 0]]
            for t in range(1, T): zs.append(a * z[:, t] + (1 - a) * zs[-1])      # temporal reliability updating
            zs = torch.stack(zs, 1)
            w = 3 * torch.softmax(zs, -1)                                        # dynamic feature weighting (mean weight = 1)
            last = G[:, -1] * w[:, -1].unsqueeze(-1)                             # (B,3,d)
            ao, _ = self.attn(last, last, last); last = self.norm(last + ao)
            feat = torch.cat([last.flatten(1), c[:, -1], m[:, -1]], -1)
            wl = w[:, -1] / 3
        elif self.mode == 'fixed':
            w = 3 * torch.softmax(self.fixed, 0)
            feat = (G[:, -1] * w[None, :, None]).flatten(1); wl = (w / 3).unsqueeze(0).expand(B, -1)
        else:                                                                    # 'concat'
            feat = G[:, -1].flatten(1); wl = torch.full((B, 3), 1 / 3, device=G.device)
        return self.out(feat), wl

class CMTNet(nn.Module):
    # CMLF -> TDRF -> multi-target heads. The bounded embedding feeds AFMM-D.
    def __init__(self, dims, d=32, e=16, coupling=True, fusion='dynamic', drop=0.1):
        super().__init__()
        self.cmlf = CMLF(dims, d, coupling, drop); self.tdrf = TDRF(d, e, mode=fusion, drop=drop); self.ln = nn.LayerNorm(e)
        mk = lambda o: nn.Sequential(nn.Linear(e, 64), nn.GELU(), nn.Dropout(drop), nn.Linear(64, o))
        self.h_dir, self.h_ret, self.h_vol = mk(3), mk(1), mk(1)
    def forward(self, xp, xt, xv):
        G, U = self.cmlf(xp, xt, xv); z, w = self.tdrf(G, U)
        emb = torch.sigmoid(self.ln(z))
        return dict(logits=self.h_dir(emb), ret=self.h_ret(emb).squeeze(-1),
                    vol=F.softplus(self.h_vol(emb).squeeze(-1)) + 1e-3, emb=emb, weights=w)

class BaselineNet(nn.Module):
    # Architecture-family PROXIES for existing methods (same inputs, same multi-target heads).
    # 'mbilstm' -> MBi-LSTM [19] (stacked BiLSTM) | 'cnn_bigru' -> BGSTO-DC [16] (dilated conv + BiGRU)
    # 'mlp' -> MOPO [17] (multi-objective MLP)     | 'transformer' -> AI-MDP [18] (Transformer encoder)
    # Replace with the exact published architectures for the final paper.
    def __init__(self, in_dim, kind, T, h=64, drop=0.1):
        super().__init__()
        self.kind = kind
        if kind == 'mbilstm':
            self.rnn = nn.LSTM(in_dim, h, num_layers=2, batch_first=True, bidirectional=True, dropout=drop); o = 2 * h
        elif kind == 'cnn_bigru':
            self.conv = nn.Sequential(nn.Conv1d(in_dim, h, 3, padding=1), nn.GELU(), nn.Conv1d(h, h, 3, padding=2, dilation=2), nn.GELU())
            self.rnn = nn.GRU(h, h, batch_first=True, bidirectional=True); o = 2 * h
        elif kind == 'transformer':
            self.inp = nn.Linear(in_dim, h); self.pos = nn.Parameter(torch.zeros(T, h))
            self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(h, 4, 2 * h, drop, batch_first=True), 2); o = h
        else:
            self.mlp = nn.Sequential(nn.Flatten(), nn.Linear(T * in_dim, 2 * h), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * h, h), nn.GELU()); o = h
        mk = lambda k: nn.Sequential(nn.Linear(o, 64), nn.GELU(), nn.Dropout(drop), nn.Linear(64, k))
        self.h_dir, self.h_ret, self.h_vol = mk(3), mk(1), mk(1)
    def forward(self, xp, xt, xv):
        x = torch.cat([xp, xt, xv], -1)
        if self.kind == 'mbilstm': z = self.rnn(x)[0][:, -1]
        elif self.kind == 'cnn_bigru': z = self.rnn(self.conv(x.transpose(1, 2)).transpose(1, 2))[0][:, -1]
        elif self.kind == 'transformer': z = self.enc(self.inp(x) + self.pos).mean(1)
        else: z = self.mlp(x)
        return dict(logits=self.h_dir(z), ret=self.h_ret(z).squeeze(-1), vol=F.softplus(self.h_vol(z).squeeze(-1)) + 1e-3, emb=None, weights=None)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
_m = CMTNet(DIMS, cfg.d, cfg.e); print('CMTNet (proposed) trainable parameters:', count_params(_m)); del _m

---
## Training and inference utilities (multi-target loss: class-weighted CE + Huber return + Huber volatility)

In [ ]:
# ===================== TRAINING UTILITIES =====================
KEYS = ('Xp', 'Xt', 'Xv', 'y_dir', 'y_ret', 'y_vol')
def make_loader(D, shuffle):
    ds = TensorDataset(*[torch.from_numpy(D[k]) for k in KEYS])
    return DataLoader(ds, batch_size=cfg.batch, shuffle=shuffle)

def mt_loss(out, yd, yr, yv):
    return (F.cross_entropy(out['logits'], yd, weight=CLASS_W.to(out['logits'].device))
            + cfg.lam_ret * F.smooth_l1_loss(out['ret'], yr) + cfg.lam_vol * F.smooth_l1_loss(out['vol'], yv))

def train_model(model, name, verbose=True):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    tl, vl = make_loader(DATA['train'], True), make_loader(DATA['val'], False)
    best, best_state, bad, hist = 1e9, None, 0, []
    for ep in range(cfg.epochs):
        model.train(); tot, n = 0.0, 0
        for xp, xt, xv, yd, yr, yv in tl:
            xp, xt, xv, yd, yr, yv = [t.to(DEVICE) for t in (xp, xt, xv, yd, yr, yv)]
            opt.zero_grad(); loss = mt_loss(model(xp, xt, xv), yd, yr, yv); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(yd); n += len(yd)
        sched.step(); model.eval(); vt, vn = 0.0, 0
        with torch.no_grad():
            for xp, xt, xv, yd, yr, yv in vl:
                xp, xt, xv, yd, yr, yv = [t.to(DEVICE) for t in (xp, xt, xv, yd, yr, yv)]
                vt += mt_loss(model(xp, xt, xv), yd, yr, yv).item() * len(yd); vn += len(yd)
        hist.append((tot / n, vt / vn))
        if verbose: print(f'[{name}] ep {ep+1:02d} train {tot/n:.4f} val {vt/vn:.4f}')
        if vt / vn < best - 1e-4: best, bad, best_state = vt / vn, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= cfg.patience: break
    model.load_state_dict(best_state); model.eval()
    return hist

@torch.no_grad()
def predict(model, D, noise_std=0.0, bs=1024, seed=0):
    model.eval(); g = torch.Generator(device='cpu').manual_seed(seed)
    P, R, V, E, W = [], [], [], [], []
    for i in range(0, len(D['y_dir']), bs):
        xs = [torch.from_numpy(D[k][i:i + bs]) for k in ('Xp', 'Xt', 'Xv')]
        if noise_std > 0: xs = [x + noise_std * torch.randn(x.shape, generator=g) for x in xs]
        o = model(*[x.to(DEVICE) for x in xs])
        P.append(torch.softmax(o['logits'], -1).cpu().numpy()); R.append(o['ret'].cpu().numpy()); V.append(o['vol'].cpu().numpy())
        if o['emb'] is not None: E.append(o['emb'].cpu().numpy()); W.append(o['weights'].cpu().numpy())
    return dict(probs=np.concatenate(P), ret=np.concatenate(R), vol=np.concatenate(V),
                emb=np.concatenate(E) if E else None, w=np.concatenate(W) if W else None)

---
## §5  Stage 5 — AFMM-D: Adaptive Fuzzy Min–Max Market Decision Network
Operates on the TDRF market embedding (values in [0,1]).
* **Hyperbox initialisation** – a new point-hyperbox is created when no same-class box can absorb a sample.
* **Fuzzy membership** – Simpson membership with sensitivity γ.
* **Market-state representation** – each hyperbox carries one of the three market states (down / stable / up).
* **Adaptive expansion** – expansion is allowed only if membership ≥ τ and size ≤ θ; θ decays across epochs and is tightened per-box by `1/(1+#contractions)`.
* **Contraction / overlap management** – Simpson's four overlap cases for boxes of different states; wide boxes are shrunk to the 5–95 % support of their assigned samples and unsupported boxes are pruned.
* **Fuzzy confidence** – from the top class membership and its margin over the runner-up.
* **Adaptive decision refinement** – `partial_update` learns new labelled samples online (used in the sequential test replay).

In [ ]:
# ===================== 5. AFMM-D =====================
class AFMMD:
    def __init__(self, n_dim, n_classes=3, theta0=0.4, theta_min=0.10, decay=0.8, gamma=6.0, tau=0.7, seed=0, max_boxes=1500):
        self.max_boxes = max_boxes
        self.n, self.k, self.theta0, self.theta_min, self.decay, self.gamma, self.tau = n_dim, n_classes, theta0, theta_min, decay, gamma, tau
        self.V = np.zeros((0, n_dim), np.float32); self.W = np.zeros((0, n_dim), np.float32)
        self.C = np.zeros(0, int); self.contr = np.zeros(0); self.theta_cur = theta0; self.rng = np.random.default_rng(seed)
    # ---- Fuzzy Membership Calculation (Simpson) ----
    def _mem(self, A):
        out = np.empty((len(A), len(self.V)), np.float32); g = self.gamma
        for s in range(0, len(A), 1024):
            a = A[s:s + 1024][:, None, :]
            hi = np.clip(g * (a - self.W[None]), 0, 1); lo = np.clip(g * (self.V[None] - a), 0, 1)
            out[s:s + 1024] = (1 - 0.5 * (hi + lo)).mean(-1)
        return out
    def _new_box(self, a, c):                                                   # Fuzzy Hyperbox Initialization
        self.V = np.vstack([self.V, a[None]]); self.W = np.vstack([self.W, a[None]])
        self.C = np.append(self.C, c); self.contr = np.append(self.contr, 0.0)
    def learn_one(self, a, c, theta=None):
        theta = self.theta_cur if theta is None else theta
        if len(self.C) == 0: return self._new_box(a, c)
        idx = np.where(self.C == c)[0]; done = -1
        if len(idx):
            m = self._mem(a[None])[0][idx]
            for kk in np.argsort(-m):                                           # Adaptive Hyperbox Expansion
                j = idx[kk]; th = theta / (1 + self.contr[j])
                nv, nw = np.minimum(self.V[j], a), np.maximum(self.W[j], a)
                if m[kk] >= self.tau and (nw - nv).sum() <= self.n * th:
                    self.V[j], self.W[j] = nv, nw; done = j; break
        if done < 0:
            if len(self.C) < self.max_boxes: self._new_box(a, c)      # box budget keeps the model compact
        else: self._contract(done)
    # ---- Hyperbox Overlap Management + Contraction ----
    def _contract(self, j):
        other = np.where(self.C != self.C[j])[0]
        if len(other) == 0: return
        ov = (np.maximum(self.V[j], self.V[other]) <= np.minimum(self.W[j], self.W[other])).all(1)
        for k in other[ov]: self._contract_pair(j, k)
    def _contract_pair(self, j, k):
        vj, wj, vk, wk = self.V[j], self.W[j], self.V[k], self.W[k]
        best, dim, cs = 1e9, -1, 0
        for i in range(self.n):
            if vj[i] < vk[i] < wj[i] < wk[i]: d, c_ = wj[i] - vk[i], 1
            elif vk[i] < vj[i] < wk[i] < wj[i]: d, c_ = wk[i] - vj[i], 2
            elif vj[i] < vk[i] <= wk[i] < wj[i]: d, c_ = min(wj[i] - vk[i], wk[i] - vj[i]), 3
            elif vk[i] < vj[i] <= wj[i] < wk[i]: d, c_ = min(wj[i] - vk[i], wk[i] - vj[i]), 4
            else: d, c_ = max(0.0, min(wj[i], wk[i]) - max(vj[i], vk[i])), 5
            if d < best: best, dim, cs = d, i, c_
        i = dim
        if cs == 1: vk[i] = wj[i] = (wj[i] + vk[i]) / 2
        elif cs == 2: vj[i] = wk[i] = (wk[i] + vj[i]) / 2
        elif cs == 3:
            if wk[i] - vj[i] > wj[i] - vk[i]: vj[i] = wk[i]
            else: wj[i] = vk[i]
        elif cs == 4:
            if wj[i] - vk[i] > wk[i] - vj[i]: vk[i] = wj[i]
            else: wk[i] = vj[i]
        else:
            mid = (max(vj[i], vk[i]) + min(wj[i], wk[i])) / 2
            if vj[i] <= vk[i]: wj[i] = vk[i] = mid
            else: wk[i] = vj[i] = mid
        self.contr[j] += 1; self.contr[k] += 1
    def n_overlaps(self):
        c = 0
        for j in range(len(self.C)):
            o = np.where(self.C[j + 1:] != self.C[j])[0] + j + 1
            if len(o): c += int((np.maximum(self.V[j], self.V[o]) < np.minimum(self.W[j], self.W[o])).all(1).sum())
        return c
    def fit(self, X, y, epochs=3, verbose=True):
        for ep in range(epochs):
            self.theta_cur = max(self.theta_min, self.theta0 * self.decay ** ep)
            for i in self.rng.permutation(len(X)): self.learn_one(X[i], int(y[i]))
            if verbose: print(f'  AFMM-D epoch {ep+1}: theta={self.theta_cur:.3f} hyperboxes={len(self.C)} contractions={int(self.contr.sum())}')
        self.shrink_and_prune(X, y); return self
    def shrink_and_prune(self, X, y, wide=0.30, min_support=2):
        M = self._mem(X); win = M.argmax(1); keep = np.ones(len(self.C), bool)
        for j in range(len(self.C)):
            sel = (win == j) & (y == self.C[j])
            if sel.sum() < min_support and len(self.C) > 50: keep[j] = False; continue
            if sel.sum() >= 5 and (self.W[j] - self.V[j]).mean() > wide:        # reduce excessively broad regions
                self.V[j] = np.percentile(X[sel], 5, axis=0); self.W[j] = np.percentile(X[sel], 95, axis=0)
        self.V, self.W, self.C, self.contr = self.V[keep], self.W[keep], self.C[keep], self.contr[keep]
    def scores_conf(self, A):                                                   # Market-State Representation + Confidence
        M = self._mem(A); S = np.zeros((len(A), self.k), np.float32)
        for c in range(self.k):
            idx = self.C == c
            if idx.any(): S[:, c] = M[:, idx].max(1)
        srt = np.sort(S, 1)[:, ::-1]; top, sec = srt[:, 0], srt[:, 1]
        conf = np.clip(0.5 * top + 0.5 * np.clip(5 * (top - sec), 0, 1), 0, 1)  # Fuzzy Confidence Estimation
        return S, conf
    def partial_update(self, A, y):                                             # Adaptive Decision Refinement (online)
        th = max(self.theta_min, self.theta_cur)
        for a, c in zip(A, y): self.learn_one(a, int(c), th)

def fit_afmm(deep_tr, deep_va, verbose=True, n_search=5000):
    # theta0 chosen on validation macro-F1 using a train subsample (fast); the best theta is then refitted on all train data
    ytr = DATA['train']['y_dir']; rng = np.random.default_rng(0)
    sub_i = rng.choice(len(ytr), min(n_search, len(ytr)), replace=False); best_th, best_f = cfg.afmm_theta_grid[0], -1
    for th in cfg.afmm_theta_grid:
        f = AFMMD(cfg.e, 3, theta0=th, gamma=cfg.afmm_gamma, tau=cfg.afmm_tau)
        f.fit(deep_tr['emb'][sub_i], ytr[sub_i], cfg.afmm_epochs, verbose=False)
        S, _ = f.scores_conf(deep_va['emb'])
        sc = precision_recall_fscore_support(DATA['val']['y_dir'], S.argmax(1), average='macro', zero_division=0)[2]
        if verbose: print(f'  theta0={th}: boxes={len(f.C)} overlaps={f.n_overlaps()} val macro-F1={sc:.4f}')
        if sc > best_f: best_th, best_f = th, sc
    f = AFMMD(cfg.e, 3, theta0=best_th, gamma=cfg.afmm_gamma, tau=cfg.afmm_tau)
    f.fit(deep_tr['emb'], ytr, cfg.afmm_epochs, verbose=verbose)
    if verbose: print(f'  final AFMM-D: theta0={best_th} hyperboxes={len(f.C)} overlaps={f.n_overlaps()}')
    return f

---
## §6  Stage 6 — Multi-Target Prediction and Fuzzy-Confidence Integration
Direction probabilities = mixture of the deep head and the normalised fuzzy class memberships (mixing weight `beta·confidence`); expected return is shrunk toward zero when confidence is low; predicted volatility is inflated by an uncertainty premium; final confidence = average of fuzzy confidence and deep-probability margin.

In [ ]:
# ===================== 6. MULTI-TARGET FORECAST =====================
def fuse_forecast(deep, S, conf):
    pf = S / (S.sum(1, keepdims=True) + 1e-9); pf[S.sum(1) < 1e-9] = 1 / 3
    mix = (cfg.beta_mix * conf)[:, None]
    probs = (1 - mix) * deep['probs'] + mix * pf                                   # Fuzzy Confidence Integration
    dconf = (probs.max(1) - 1 / 3) / (2 / 3)
    return dict(probs=probs, dir=probs.argmax(1), ret=deep['ret'] * (0.5 + 0.5 * conf),
                vol=deep['vol'] * (1 + 0.25 * (1 - conf)), conf=0.5 * conf + 0.5 * dconf)   # Multi-Target Prediction Fusion

def deep_only_forecast(deep):                                                      # conventional prediction layer
    conf = (deep['probs'].max(1) - 1 / 3) / (2 / 3)
    return dict(probs=deep['probs'], dir=deep['probs'].argmax(1), ret=deep['ret'], vol=deep['vol'], conf=conf)

def forecast_static(deep, fmm):
    S, conf = fmm.scores_conf(deep['emb']); return fuse_forecast(deep, S, conf)

def forecast_online(deep, fmm, D):
    # Sequential replay: label of the sample dated d is revealed at d+1 close, so it is learned just before predicting d+1.
    f = copy.deepcopy(fmm); dates = D['date']; order = np.argsort(dates, kind='stable'); ud = np.unique(dates)
    S = np.zeros((len(dates), 3), np.float32); C = np.zeros(len(dates), np.float32); prev = None
    for d in ud:
        cur = np.where(dates == d)[0]
        if prev is not None: f.partial_update(deep['emb'][prev], D['y_dir'][prev])
        S[cur], C[cur] = f.scores_conf(deep['emb'][cur]); prev = cur
    return fuse_forecast(deep, S, C)

---
## §7  Stage 7 — Risk-Aware Financial Decision Generation
Prediction-state mapping (down/stable/up → bearish/neutral/bullish) → return–risk integration (`ret/vol`) → confidence filter (threshold = validation-set quantile, no test leakage) → **BUY / HOLD / SELL** → volatility-targeted, confidence-scaled position size → exposure control (haircut when predicted volatility exceeds the train 80th percentile).
Semantics: BUY → target the computed size; SELL → exit; HOLD → keep the previous target.

In [ ]:
# ===================== 7. SIGNALS =====================
def generate_signals(fc, conf_min):
    state = fc['dir'] - 1                                              # Prediction-State Mapping: -1 bearish, 0 neutral, +1 bullish
    ra = fc['ret'] / np.maximum(fc['vol'], 1e-3)                       # Return-Risk Integration
    strong = fc['conf'] >= conf_min                                    # Confidence-Aware Decision Filtering
    buy = (state == 1) & strong & (fc['ret'] > 0) & (ra >= cfg.ra_min)
    sell = (state == -1) & strong
    S = np.where(buy, 1, np.where(sell, -1, 0))                        # Trading Signal Generation
    size = np.clip(cfg.target_vol / np.maximum(fc['vol'], 1e-3), 0, 1.5) * fc['conf'] * cfg.base_size   # Risk-Adjusted Position Sizing
    size = np.where(fc['vol'] > VOL_HI, 0.5 * size, size)              # Exposure Control (high-volatility haircut)
    return S.astype(int), size.astype(float)

---
## §8  Stage 8 — Strategic Financial Management (portfolio engine)
Per trading day: signals → target weights (risk-constrained: per-asset cap, gross-exposure cap, remainder in cash) → **capital-preservation control** (exposure scaled down when portfolio drawdown or rolling volatility breaches limits) → **periodic rebalancing** every `rebalance_every` days, with **dynamic position adjustment** in between (positions change immediately when an asset's signal changes or its target moves by more than `adj_band`) → transaction costs on turnover → **risk-exposure monitoring** (rolling volatility, concentration HHI, drawdown, exposure).
Timing: predictions use data up to the close of day *d*; the position earns the *d→d+1* return.

In [ ]:
# ===================== 8. PORTFOLIO MANAGEMENT =====================
def build_panel(D, fc, S, Z):
    df = pd.DataFrame(dict(date=D['date'], tid=D['tid'], S=S, Z=Z, R=np.expm1(D['y_ret'] / 100.0)))
    dates = np.sort(df['date'].unique()); n_t = len(TICK)
    pv = lambda col, fill: df.pivot(index='date', columns='tid', values=col).reindex(dates).reindex(columns=range(n_t)).fillna(fill).values
    return dates, pv('S', 0).astype(int), pv('Z', 0.0), pv('R', 0.0)

def backtest(S, Z, R, cost_bps=None):
    cost = (cfg.cost_bps if cost_bps is None else cost_bps) / 1e4
    D_, N = S.shape; tgt_prev = np.zeros(N); w_prev = np.zeros(N)
    equity, peak = 1.0, 1.0; port, rec = [], []
    for t in range(D_):
        tgt = np.where(S[t] == 1, Z[t], np.where(S[t] == -1, 0.0, tgt_prev))                     # BUY / SELL / HOLD
        tgt = np.minimum(np.nan_to_num(tgt), cfg.max_w)                                           # Risk-Constrained Allocation (asset cap)
        tot = tgt.sum()
        if tot > cfg.max_gross: tgt = tgt * cfg.max_gross / tot                                   # gross exposure cap, rest in cash
        dd = equity / peak - 1
        rv = np.std(port[-20:]) * np.sqrt(252) if len(port) >= 10 else 0.0
        scale = cfg.preserve_scale if (dd < -cfg.dd_trigger or rv > cfg.pvol_limit) else 1.0     # Capital Preservation Control
        eff = tgt * scale
        if t % cfg.rebalance_every == 0: w = eff                                                  # Portfolio Rebalancing
        else: w = np.where((S[t] != 0) | (np.abs(eff - w_prev) > cfg.adj_band), eff, w_prev)      # Dynamic Position Adjustment
        turn = np.abs(w - w_prev).sum(); r = float((w * R[t]).sum() - cost * turn)
        port.append(r); equity *= (1 + r); peak = max(peak, equity); tgt_prev = tgt
        ws = w.sum(); rec.append((ws, float(((w / ws) ** 2).sum()) if ws > 1e-9 else 0.0, equity / peak - 1, turn))
        w_prev = w
    rec = np.array(rec); ser = pd.Series(port)
    return dict(ret=np.array(port), exposure=rec[:, 0], hhi=rec[:, 1], drawdown=rec[:, 2], turnover=rec[:, 3],
                roll_vol=(ser.rolling(20).std() * np.sqrt(252)).values)          # Risk Exposure Monitoring

def perf_metrics(r):
    r = np.asarray(r); eq = np.cumprod(1 + r); peak = np.maximum.accumulate(eq)
    ds = np.sqrt(np.mean(np.minimum(r, 0) ** 2)) + 1e-12
    return dict(CumRet=eq[-1] - 1, AnnRet=eq[-1] ** (252 / len(r)) - 1, Sharpe=r.mean() / (r.std() + 1e-12) * np.sqrt(252),
                Sortino=r.mean() / ds * np.sqrt(252), MaxDD=(eq / peak - 1).min(), PortVol=r.std() * np.sqrt(252))

---
## Train the proposed model, AFMM-D, ablation variants and the baselines
Every model is trained with the same data, loss and early stopping; every forecast goes through the *same* signal and portfolio engine.

In [ ]:
# ===================== TRAIN ALL MODELS =====================
BUNDLES, HIST = {}, {}
def train_cmt(name, coupling, fusion):
    set_seed(); m = CMTNet(DIMS, cfg.d, cfg.e, coupling, fusion, cfg.drop)
    HIST[name] = train_model(m, name, verbose=(name == 'Proposed'))
    dtr, dva = predict(m, DATA['train']), predict(m, DATA['val'])
    print(f'[{name}] fitting AFMM-D ...'); fmm = fit_afmm(dtr, dva, verbose=(name == 'Proposed'))
    BUNDLES[name] = dict(model=m, fmm=fmm, kind='cmt', cfg=(coupling, fusion))

def train_base(name, kind):
    set_seed(); m = BaselineNet(sum(DIMS), kind, cfg.window, drop=cfg.drop)
    HIST[name] = train_model(m, name, verbose=False); BUNDLES[name] = dict(model=m, fmm=None, kind='base')

t0 = time.time()
train_cmt('Proposed',               True,  'dynamic')     # CMLF + TDRF + AFMM-D
train_cmt('Abl: no CMLF coupling',  False, 'dynamic')
train_cmt('Abl: TDRF=concat',       True,  'concat')
train_cmt('Abl: TDRF=fixed-weight', True,  'fixed')
train_cmt('Indep+concat',           False, 'concat')
for nm, kd in [('MBi-LSTM [19]', 'mbilstm'), ('BGSTO-DC [16]*', 'cnn_bigru'), ('MOPO [17]*', 'mlp'), ('AI-MDP [18]*', 'transformer')]:
    train_base(nm, kd)
print('Total training time: %.1f min' % ((time.time() - t0) / 60))
print('* = architecture-family proxy; MBi-LSTM is also a generic stacked BiLSTM - replace with published models for the paper.')

---
## §10 (part 1)  Evaluation helpers — Stage 10 metrics
Classification (Accuracy, macro Precision/Recall/F1), regression (MAE, RMSE, MAPE, Directional Accuracy; return/volatility in **percent** units), financial (Cumulative Return, Sharpe, Sortino, Max Drawdown, Portfolio Volatility, transaction-cost analysis) and efficiency (parameters, FLOPs, inference time).
MAPE of returns is ill-defined near zero, so the denominator is floored at 0.25 %; the volatility MAPE is the more meaningful one.

In [ ]:
# ===================== EVALUATION HELPERS =====================
def cls_reg_metrics(D, fc):
    y, p = D['y_dir'], fc['dir']
    P, R, F1, _ = precision_recall_fscore_support(y, p, average='macro', zero_division=0)
    err = fc['ret'] - D['y_ret']
    return dict(Acc=accuracy_score(y, p), Prec=P, Rec=R, F1=F1, MAE=np.mean(np.abs(err)), RMSE=np.sqrt(np.mean(err ** 2)),
                MAPE_ret=np.mean(np.abs(err) / np.maximum(np.abs(D['y_ret']), 0.25)) * 100,
                MAPE_vol=np.mean(np.abs(fc['vol'] - D['y_vol']) / np.maximum(D['y_vol'], 1e-3)) * 100,
                DirAcc=np.mean(np.sign(fc['ret']) == np.sign(D['y_ret'])))

def sub(D, mask): return {k: (v[mask] if isinstance(v, np.ndarray) and len(v) == len(mask) else v) for k, v in D.items()}
def sub_fc(fc, mask): return {k: v[mask] for k, v in fc.items()}

def get_forecast(bundle, D, use_afmm=True, noise=0.0, online=False):
    deep = predict(bundle['model'], D, noise_std=noise)
    if bundle['kind'] == 'base' or not use_afmm: return deep_only_forecast(deep), deep
    return (forecast_online(deep, bundle['fmm'], D) if online else forecast_static(deep, bundle['fmm'])), deep

def conf_threshold(bundle, use_afmm):
    fv, _ = get_forecast(bundle, DATA['val'], use_afmm); return float(np.quantile(fv['conf'], cfg.conf_q))

def run_financial(D, fc, conf_min, cost_bps=None):
    S, Z = generate_signals(fc, conf_min); dates, Sp, Zp, Rp = build_panel(D, fc, S, Z)
    bt = backtest(Sp, Zp, Rp, cost_bps); bt['dates'] = dates; bt['signals'] = S
    return bt, (Sp, Zp, Rp)

def full_eval(name, use_afmm=True, online=False, label=None):
    b = BUNDLES[name]; D = DATA['test']; cm = conf_threshold(b, use_afmm)
    fc, deep = get_forecast(b, D, use_afmm, online=online); bt, panel = run_financial(D, fc, cm)
    row = dict(Model=label or name, **cls_reg_metrics(D, fc), **perf_metrics(bt['ret']), AvgTurnover=bt['turnover'].mean(),
               AvgExposure=bt['exposure'].mean(), Params=count_params(b['model']))
    return row, dict(fc=fc, deep=deep, bt=bt, panel=panel, conf_min=cm)

def equal_weight_benchmark():
    D = DATA['test']; df = pd.DataFrame(dict(date=D['date'], tid=D['tid'], R=np.expm1(D['y_ret'] / 100.0)))
    return df.pivot(index='date', columns='tid', values='R').mean(1).values       # equal-weight, daily rebalanced, zero cost

---
## §10 (part 2) + main comparison — Proposed vs existing methods
Proposed = CMLF + TDRF + AFMM-D (static test forecast). `online=True` adds the adaptive decision refinement replay (reported separately).

In [ ]:
# ===================== MAIN COMPARISON =====================
EV, ROWS = {}, []
for nm, lab, afmm in [('MBi-LSTM [19]', None, False), ('BGSTO-DC [16]*', None, False), ('MOPO [17]*', None, False), ('AI-MDP [18]*', None, False),
                      ('Proposed', 'Proposed (CMLF+TDRF+AFMM-D)', True)]:
    row, ev = full_eval(nm, afmm, label=lab); ROWS.append(row); EV[row['Model']] = ev
row, ev = full_eval('Proposed', True, online=True, label='Proposed + online refinement'); ROWS.append(row); EV[row['Model']] = ev
ew = equal_weight_benchmark(); ROWS.append(dict(Model='Equal-weight buy&hold', **perf_metrics(ew)))
RESULTS = pd.DataFrame(ROWS).set_index('Model')
cls_cols = ['Acc', 'Prec', 'Rec', 'F1', 'MAE', 'RMSE', 'MAPE_ret', 'MAPE_vol', 'DirAcc']
fin_cols = ['CumRet', 'AnnRet', 'Sharpe', 'Sortino', 'MaxDD', 'PortVol', 'AvgTurnover', 'AvgExposure']
print('=== Prediction performance (test) ==='); display(RESULTS[cls_cols].round(4))
print('=== Financial performance (test, %d bps cost) ===' % cfg.cost_bps); display(RESULTS[fin_cols].round(4))

---
## §9  Stage 9 — Robustness and Ablation Analysis

In [ ]:
# ===================== 9. ROBUSTNESS =====================
bp = BUNDLES['Proposed']; Dt = DATA['test']; ev_p = EV['Proposed (CMLF+TDRF+AFMM-D)']; fc_p = ev_p['fc']

# --- Temporal generalisation: consecutive unseen test blocks ---
ud = np.unique(Dt['date']); blocks = np.array_split(ud, 4); rows = []
for i, bl in enumerate(blocks):
    mk = np.isin(Dt['date'], bl); m = cls_reg_metrics(sub(Dt, mk), sub_fc(fc_p, mk))
    msk_d = np.isin(ev_p['bt']['dates'], bl); pm = perf_metrics(ev_p['bt']['ret'][msk_d])
    rows.append(dict(Block=f'T{i+1} ({str(bl[0])[:10]}..{str(bl[-1])[:10]})', Acc=m['Acc'], F1=m['F1'], DirAcc=m['DirAcc'], MAE=m['MAE'], Sharpe=pm['Sharpe'], MaxDD=pm['MaxDD']))
print('=== Temporal Generalization Testing ==='); display(pd.DataFrame(rows).set_index('Block').round(4))

# --- Market-regime testing (VIX / volatility-index terciles from TRAIN quantiles) ---
reg = np.where(Dt['vix_raw'] <= REG_Q[0], 'stable', np.where(Dt['vix_raw'] <= REG_Q[1], 'moderate', 'volatile')); rows = []
for r in ['stable', 'moderate', 'volatile']:
    mk = reg == r
    if mk.sum() > 30:
        m = cls_reg_metrics(sub(Dt, mk), sub_fc(fc_p, mk)); rows.append(dict(Regime=r, N=int(mk.sum()), Acc=m['Acc'], F1=m['F1'], DirAcc=m['DirAcc'], MAE=m['MAE'],
                                                                                   MAPE_vol=m['MAPE_vol'], MeanConf=fc_p['conf'][mk].mean(), MeanTDRFw_price=ev_p['deep']['w'][mk, 0].mean(),
                                                                                   MeanTDRFw_tech=ev_p['deep']['w'][mk, 1].mean(), MeanTDRFw_vol=ev_p['deep']['w'][mk, 2].mean()))
print('=== Market-Regime Testing (note TDRF group weights per regime) ==='); display(pd.DataFrame(rows).set_index('Regime').round(4))

# --- Noise perturbation (Gaussian noise on normalised inputs; AFMM-D not re-fitted) ---
rows = []
for s in cfg.noise_levels:
    fc, _ = get_forecast(bp, Dt, True, noise=s); m = cls_reg_metrics(Dt, fc)
    fcb, _ = get_forecast(BUNDLES['MBi-LSTM [19]'], Dt, False, noise=s); mb = cls_reg_metrics(Dt, fcb)
    rows.append(dict(sigma=s, Prop_Acc=m['Acc'], Prop_F1=m['F1'], Prop_MAE=m['MAE'], MBiLSTM_Acc=mb['Acc'], MBiLSTM_F1=mb['F1'], MBiLSTM_MAE=mb['MAE']))
NOISE = pd.DataFrame(rows).set_index('sigma'); print('=== Noise Perturbation Testing ==='); display(NOISE.round(4))

In [ ]:
# ===================== ABLATION + MODULE-WISE COMPARISON =====================
ABL = [('Full: CMLF + TDRF + AFMM-D',           'Proposed',               True),
       ('CMLF ablation (no coupling)',          'Abl: no CMLF coupling',  True),
       ('TDRF ablation (concat fusion)',        'Abl: TDRF=concat',       True),
       ('TDRF ablation (fixed-weight fusion)',  'Abl: TDRF=fixed-weight', True),
       ('AFMM-D ablation (conventional layer)', 'Proposed',               False),
       ('Module-wise: CMLF+TDRF (no AFMM-D)',   'Proposed',               False),
       ('Module-wise: CMLF only (concat, no AFMM-D)', 'Abl: TDRF=concat', False),
       ('Module-wise: TDRF only (no coupling, no AFMM-D)', 'Abl: no CMLF coupling', False),
       ('Module-wise: independent modules + concat',  'Indep+concat',     False),
       ('Module-wise: independent + concat + AFMM-D', 'Indep+concat',     True)]
rows = []
for lab, nm, af in ABL:
    r, e = full_eval(nm, af, label=lab); r['RollAccStd'] = None
    # Prediction stability: std of accuracy over 20 rolling test windows
    ud_ = np.unique(Dt['date']); acc = []
    for ch in np.array_split(ud_, 20):
        mk = np.isin(Dt['date'], ch); acc.append(accuracy_score(Dt['y_dir'][mk], e['fc']['dir'][mk]))
    r['RollAccStd'] = float(np.std(acc)); rows.append(r)
ABLDF = pd.DataFrame(rows).set_index('Model')
print('=== Ablation / Module-wise comparison (test) ===')
display(ABLDF[['Acc', 'F1', 'DirAcc', 'MAE', 'CumRet', 'Sharpe', 'MaxDD', 'RollAccStd']].round(4))

In [ ]:
# ===================== PLOTS =====================
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
d_ = pd.to_datetime(ev_p['bt']['dates'])
for k, ev in EV.items():
    ax[0, 0].plot(d_, np.cumprod(1 + ev['bt']['ret']), label=k, lw=2.2 if k.startswith('Proposed (') else 1)
ax[0, 0].plot(d_, np.cumprod(1 + ew), 'k--', label='Equal-weight'); ax[0, 0].set_title('Equity curves (test)'); ax[0, 0].legend(fontsize=7)
for k, ev in EV.items(): ax[0, 1].plot(d_, ev['bt']['drawdown'], label=k)
ax[0, 1].set_title('Drawdown'); ax[1, 0].plot(d_, ev_p['bt']['exposure'], label='gross exposure'); ax[1, 0].plot(d_, ev_p['bt']['hhi'], label='concentration (HHI)')
ax[1, 0].plot(d_, ev_p['bt']['roll_vol'], label='rolling 20d vol (ann.)'); ax[1, 0].legend(); ax[1, 0].set_title('Risk-exposure monitoring (Proposed)')
ax[1, 1].plot(*zip(*HIST['Proposed'])); ax[1, 1].set_title('Proposed: train / val loss'); ax[1, 1].legend(['train', 'val'])
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ABLDF['Sharpe'].iloc[:5].plot.barh(ax=ax[0]); ax[0].set_title('Ablation: Sharpe'); ax[0].invert_yaxis()
ABLDF['F1'].iloc[:5].plot.barh(ax=ax[1]); ax[1].set_title('Ablation: macro-F1'); ax[1].invert_yaxis()
NOISE[['Prop_Acc', 'MBiLSTM_Acc']].plot(ax=ax[2], marker='o'); ax[2].set_title('Accuracy vs input noise sigma')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(fc_p['conf'], 40); ax[0].axvline(ev_p['conf_min'], c='r'); ax[0].set_title('Fuzzy prediction confidence (red = filter threshold)')
cm_ = confusion_matrix(Dt['y_dir'], fc_p['dir'], normalize='true'); im = ax[1].imshow(cm_, cmap='Blues'); ax[1].set_title('Confusion matrix (row-normalised)')
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(['down', 'stable', 'up']); ax[1].set_yticks(range(3)); ax[1].set_yticklabels(['down', 'stable', 'up'])
for i in range(3):
    for j in range(3): ax[1].text(j, i, f'{cm_[i, j]:.2f}', ha='center', va='center')
plt.tight_layout(); plt.show()

---
## §10 (part 3)  Transaction-cost analysis and computational efficiency

In [ ]:
# ===================== TRANSACTION COST + EFFICIENCY =====================
rows = []
for k, ev in EV.items():
    for c in cfg.cost_grid:
        bt, _ = run_financial(DATA['test'], ev['fc'], ev['conf_min'], cost_bps=c); pm = perf_metrics(bt['ret'])
        rows.append(dict(Model=k, cost_bps=c, CumRet=pm['CumRet'], Sharpe=pm['Sharpe'], MaxDD=pm['MaxDD'], AvgTurnover=bt['turnover'].mean()))
COST = pd.DataFrame(rows)
print('=== Transaction-Cost Analysis ==='); display(COST.pivot(index='Model', columns='cost_bps', values='Sharpe').round(3).rename_axis(columns='Sharpe @ bps'))
display(COST.pivot(index='Model', columns='cost_bps', values='CumRet').round(3).rename_axis(columns='CumRet @ bps'))

def flops_of(model):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        xs = [torch.from_numpy(DATA['test'][k][:1]).to(DEVICE) for k in ('Xp', 'Xt', 'Xv')]; model.eval()
        with FlopCounterMode(display=False) as fcm, torch.no_grad(): model(*xs)
        return fcm.get_total_flops()
    except Exception as ex:
        return float('nan')

@torch.no_grad()
def infer_time_ms(bundle, use_afmm, reps=100):
    xs = [torch.from_numpy(DATA['test'][k][:1]).to(DEVICE) for k in ('Xp', 'Xt', 'Xv')]; m = bundle['model']; m.eval()
    for _ in range(10): m(*xs)
    ts = []
    for _ in range(reps):
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); o = m(*xs)
        if use_afmm: bundle['fmm'].scores_conf(o['emb'].cpu().numpy())
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))

rows = []
for nm, af in [('MBi-LSTM [19]', False), ('BGSTO-DC [16]*', False), ('MOPO [17]*', False), ('AI-MDP [18]*', False), ('Proposed', True)]:
    b = BUNDLES[nm]; fl = flops_of(b['model'])
    if af: fl += len(b['fmm'].C) * cfg.e * 6                                  # hyperbox membership cost estimate
    rows.append(dict(Model=nm if nm != 'Proposed' else 'Proposed (incl. AFMM-D)', Params=count_params(b['model']), FLOPs=fl, Hyperboxes=len(b['fmm'].C) if af else 0,
                     InferenceMs=infer_time_ms(b, af)))
EFF = pd.DataFrame(rows).set_index('Model'); print('=== Computational efficiency ==='); display(EFF)

---
## Final summary tables and export

In [ ]:
# ===================== FINAL SUMMARY + EXPORT =====================
FINAL = RESULTS.join(EFF[['Params', 'FLOPs', 'InferenceMs']].rename(index={'Proposed (incl. AFMM-D)': 'Proposed (CMLF+TDRF+AFMM-D)'}), rsuffix='_e')
print('=== FINAL: prediction + financial + efficiency ==='); display(FINAL[['Acc', 'F1', 'DirAcc', 'MAE', 'RMSE', 'CumRet', 'Sharpe', 'Sortino', 'MaxDD', 'PortVol', 'InferenceMs']].round(4))
os.makedirs('outputs', exist_ok=True)
RESULTS.to_csv('outputs/main_results.csv'); ABLDF.to_csv('outputs/ablation.csv'); NOISE.to_csv('outputs/noise.csv'); COST.to_csv('outputs/cost_analysis.csv'); EFF.to_csv('outputs/efficiency.csv')
torch.save({k: v['model'].state_dict() for k, v in BUNDLES.items()}, 'outputs/models.pt')
print('Saved to ./outputs   | synthetic data used:', USED_SYNTH)
if USED_SYNTH: print('!! WARNING: these numbers come from SYNTHETIC data and are meaningless for reporting.')

---
### Practical notes
* **Statistical caution:** daily direction accuracy on liquid equities is typically only slightly above chance. Run several seeds (`SEED`), report mean ± std, and use walk-forward re-training for the final paper.
* **Look-ahead safety:** winsorisation, normalisation, dead-zone threshold, confidence threshold, regime cut-offs and class weights are all fitted on train/validation only; targets are purged at split edges.
* **Baselines:** proxies unless you replace `BaselineNet` with the exact [16]–[19] architectures.
* **Execution assumptions:** trades at the close of day *d*, no slippage beyond the cost in bps, long-only with cash, ignores dividends beyond adjusted prices.